In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
import multiprocessing
from datetime import datetime
import glob

# Load graph

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

# Gerichtsentscheidungen pro Jahr

In [ ]:
decision_keys = [
    n 
    for n, bipartite in G.nodes(data="bipartite") 
    if bipartite == 'decision'
]
len(decision_keys)

In [ ]:
def heatmap_bars_chart(dates, title=None, color="#1f77b4", color_scheme="blues"):
    binned_weeks = Counter([
        tuple(datetime.strptime(d, '%Y-%m-%d').isocalendar()[:2])
        for d in dates
    ])
    
    df_binned_year_weeks = pd.DataFrame(
        [(k[0], k[1], v) for k, v in binned_weeks.most_common()], 
        columns=['Jahr', 'Woche', 'Deutschland']
    )
    
    df_binned_year_weeks = df_binned_year_weeks[df_binned_year_weeks.Jahr<=2020]
    
    chart_heatmap = alt.Chart(
        df_binned_year_weeks,
    #     title='Entscheidungen'
    ).mark_rect().encode(
        x='Jahr:O',
        y='Woche:O',
        color=alt.Color(
            'Deutschland:Q',
            scale=alt.Scale(scheme=color_scheme),
            legend=alt.Legend(orient='none', legendY=0, legendX=233)
        ),
    ).properties(height=450, width=220)

    df_binned_weeks = df_binned_year_weeks\
        .groupby('Woche')[['Deutschland']]\
        .agg('sum')\
        .reset_index()
    
    df_binned_years = df_binned_year_weeks\
        .groupby('Jahr')[['Deutschland']]\
        .agg('sum')\
        .reset_index()

    chart_weeks = alt.Chart(df_binned_weeks).mark_bar(color=color).encode(
        alt.Y('Woche:O', axis=None),
        alt.X('Deutschland:Q', title=None),
    ).properties(width=200, height=450)

    chart_years = alt.Chart(df_binned_years).mark_bar(color=color).encode(
        alt.X('Jahr:O', axis=None),
        alt.Y('Deutschland:Q', title=None, axis=alt.Axis(tickCount=6, tickMinStep=2),),
    ).properties(height=60, width=220)
    
    title_attrs = dict(title=title) if title else {}
    chart = alt.vconcat(chart_years, alt.hconcat(chart_heatmap, chart_weeks), **title_attrs)
    chart = chart.configure_concat(
        spacing=5,
    ).configure_legend(
        gradientLength=45,
    )
    return chart, df_binned_year_weeks

In [ ]:
dates = list(nx.get_node_attributes(G, 'datum').values())

In [ ]:
chart, df_binned_year_weeks = heatmap_bars_chart(dates, None, "#1f77b4", "blues")
save_chart(chart, 'makro_decisions_count_week')

In [ ]:
chart, df_binned_year_weeks = heatmap_bars_chart(dates, None, "black", "greys")
save_chart(chart, 'makro_decisions_count_week_graycolor')

In [ ]:
diss_data(
    'makro_de_decisions_count',
    de_num_format(f'{df_binned_year_weeks.Deutschland.sum():,}')
)
diss_data(
    'makro_de_decisions_year_count_max',
    de_num_format(f"{df_binned_year_weeks.groupby('Jahr').sum().Deutschland.max():,}")
)
diss_data(
    'makro_de_decisions_year_count_min',
    de_num_format(f"{df_binned_year_weeks.groupby('Jahr').sum().Deutschland.min():,}")
)


In [ ]:
gmsogb = [x for x, g in G.nodes(data='gericht') if g == 'GmSOGB']
assert len(gmsogb) == 2
gmsogb

In [ ]:
gericht_dict = nx.get_node_attributes(G, 'gericht')
datum_dict = nx.get_node_attributes(G, 'datum')
spruchkoerper_dict = nx.get_node_attributes(G, 'spruchkoerper')

In [ ]:
gericht_counter = Counter([x for x in gericht_dict.values()]).most_common()
gericht_counter

In [ ]:
assert gericht_counter[0][0] == 'BGH'
diss_data(
    'makro_de_decisions_gericht_count_max',
    de_num_format(f"{gericht_counter[0][1]:,}")
)
assert gericht_counter[-2][0] == 'BVerfG'
diss_data(
    'makro_de_decisions_gericht_count_BVerfG',
    de_num_format(f"{gericht_counter[-2][1]:,}")
)

In [ ]:
fileList = glob.glob(f'{data_figures_path}/makro_decisions_count_week_gericht_*')
for f in fileList:
    os.remove(f)
    
for gericht, _ in gericht_counter:
    dates = [date for k, date in datum_dict.items() if gericht_dict[k] == gericht]
    chart, _ = heatmap_bars_chart(dates, f'{gericht} ({len(dates)})')
    save_chart(chart, f'makro_decisions_count_week_gericht_{gericht}')

In [ ]:
df_years_gericht = pd.DataFrame({'Jahr': []})
for gericht, _ in gericht_counter:
    dates = Counter([
        date[:4] 
        for k, date in datum_dict.items() 
        if gericht_dict[k] == gericht
    ]).most_common()
    df = pd.DataFrame(dates, columns=['Jahr', gericht])
    df_years_gericht = pd.merge(df_years_gericht, df, on='Jahr', how='outer')

In [ ]:
df_years_gericht

In [ ]:
chart1 = alt.Chart(df_years_gericht).transform_fold(
    [x[0] for x in gericht_counter],
).transform_filter(
    (alt.datum.Jahr <= 2020)
).mark_line(size=2).encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', title=f'Entscheidungen'),
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(scheme='tableau10'),
    ),
).properties(width=400)
chart2 = chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Gericht', 
        ),
        scale=alt.Scale(scheme='tableau10'),
    ),
    alt.Shape('key:N', scale=alt.Scale(range=['square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right', 'circle'])),
)
chart = (chart1 + chart2).resolve_scale(strokeDash="independent", shape='independent', color='independent').configure_point(size=70)
save_chart(chart, 'makro_decisions_count_gericht_year')

In [ ]:
chart1_grayscale = chart1.encode(
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
chart2_grayscale = chart2.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title='Gericht'),
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
chart_grayscale = (chart1_grayscale + chart2_grayscale).resolve_scale(strokeDash="independent", shape='independent', color='independent').configure_point(size=70)
save_chart(chart_grayscale, 'makro_decisions_count_gericht_year_graycolor')

In [ ]:
df_years_gericht_erledigung = pd.read_csv('../bundesgerichte_erledigungen.csv', index_col='Jahr', dtype=float)

In [ ]:
df_quote = pd.DataFrame([
    [
        years_gericht/years_gericht_erledigt
        for years_gericht, years_gericht_erledigt in zip(
            df_years_gericht[colname],
            df_years_gericht_erledigung[colname]
        )
    ]
    for colname in df_years_gericht_erledigung
], columns=list(range(2010, 2019+1)))
df_quote = df_quote.T
df_quote.columns = df_years_gericht_erledigung.columns

In [ ]:
chart1 = alt.Chart(df_quote.reset_index()).transform_fold(
    df_quote.columns.to_list(),
).mark_line(size=2).encode(
    alt.X('index:O', title='Jahr'),
    alt.Y('value:Q', title=['Enthaltene Entscheidungen /',' Erledigte Verfahren'], scale=alt.Scale(domain=(0, 0.6)), axis=alt.Axis(tickCount=7)),
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(scheme='tableau10')
    ),
)
chart2 = chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Gericht', 
        ),
        scale=alt.Scale(scheme='tableau10'),
    ),
    alt.Shape('key:N', scale=alt.Scale(range=['square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right','circle'])),
)
chart = (chart1 + chart2).resolve_scale(shape='independent', color='independent').properties(width=400).configure_point(size=70)
save_chart(chart, 'makro_decisions_count_gericht_year_quote')

In [ ]:
chart1_graycolor = chart1.encode(
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
chart2_graycolor = chart2.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title='Gericht'),
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
chart_graycolor = (chart1_graycolor + chart2_graycolor).resolve_scale(strokeDash="independent", shape='independent', color='independent').configure_point(size=70)
save_chart(chart_graycolor, 'makro_decisions_count_gericht_year_quote_graycolor')

In [ ]:
df_gericht = pd.DataFrame(df_years_gericht[df_years_gericht.columns[1:]].sum()).reset_index()
df_gericht.columns = ['Gericht', 'Entscheidungen']
df_gericht = df_gericht.sort_values('Entscheidungen', ascending=False)
df_gericht['position'] = [df_gericht.Entscheidungen[:idx].sum() + df_gericht.Entscheidungen[idx]/2 for idx in range(len(df_gericht))]

bars=alt.Chart().mark_bar().encode(
    alt.X('Entscheidungen', title='Entscheidungen', scale=alt.Scale(nice=False)),
    alt.Color('Gericht', title=None, legend=None),
    alt.Order(
      # Sort the segments of the bars by this field
      'Entscheidungen',
      sort='descending'
    )
)

text=alt.Chart().mark_text(align='left', baseline='middle', angle=270, dx=13).encode(
    alt.X('position'),
    alt.Text('Gericht'),
)

chart = alt.layer(bars, text, data=df_gericht).properties(width=400, height=20)
save_chart(chart, 'makro_decisions_count_gericht')

In [ ]:
bars_graycolor = alt.Chart().mark_bar().encode(
    alt.X('Entscheidungen', title='Entscheidungen', scale=alt.Scale(nice=False)),
    alt.Color('Gericht', title=None, legend=None, scale=alt.Scale(domain=df_gericht.Gericht.to_list(), range=["black", "#999"])),
    alt.Order(
      # Sort the segments of the bars by this field
      'Entscheidungen',
      sort='descending'
    )
)
chart_graycolor = alt.layer(bars_graycolor, text, data=df_gericht).properties(width=400, height=20)
save_chart(chart_graycolor, 'makro_decisions_count_gericht_graycolor')

In [ ]:
spruchkoerper_counter = Counter([
    (gericht_dict[k], spruchkoerper_dict[k])
    for k in spruchkoerper_dict.keys()
]).most_common(20)
spruchkoerper_counter

In [ ]:
fileList = glob.glob(f'{data_figures_path}/makro_decisions_count_week_spruchkoerper_*')
for f in fileList:
    os.remove(f)
    
for idx, v in enumerate(spruchkoerper_counter):
    tup, _ = v
    gericht, spruchkoerper = tup
    dates = [date 
             for k, date in datum_dict.items() 
             if gericht_dict[k] == gericht and
             spruchkoerper_dict[k] == spruchkoerper
            ]
    chart, _ = heatmap_bars_chart(dates, f'{gericht}/{spruchkoerper} ({len(dates)})')
    save_chart(chart, f'makro_decisions_count_week_spruchkoerper_{idx}')

In [ ]:
df_years_spruchkoerper = pd.DataFrame({'Jahr': []})
for tup, _ in spruchkoerper_counter:
    gericht, spruchkoerper = tup
    dates = Counter([
        date[:4] 
        for k, date in datum_dict.items() 
        if gericht_dict[k] == gericht and
        spruchkoerper_dict[k] == spruchkoerper
    ]).most_common()
    df = pd.DataFrame(dates, columns=['Jahr', f'{gericht}: {spruchkoerper}'])
    df_years_spruchkoerper = pd.merge(df_years_spruchkoerper, df, on='Jahr', how='outer')

In [ ]:
chart = alt.Chart(df_years_spruchkoerper).transform_fold(
    [x.replace('.','\.') for x in df_years_spruchkoerper.columns if x != 'Jahr'],
).transform_filter(
    (alt.datum.Jahr <= 2020)
).mark_line(point=True).encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', title=f'Entscheidungen'),
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Gericht', 
        ),
        scale=alt.Scale(scheme='tableau20')
    ),
).properties(width=400).configure_point(size=30)
save_chart(chart, 'makro_decisions_count_spruchkoerper_year')

# Tokens

In [ ]:
node_docs = [n for n, t in G.nodes(data='type') if t == 'document']
tokens = [G.nodes[n]['tokens_n'] for n in node_docs]
dates = [G.nodes[n]['datum'][:4] for n in node_docs]
gericht = [G.nodes[n]['gericht'] for n in node_docs]

In [ ]:
df_raw = pd.DataFrame({'Datum': dates, 'tokens': tokens, 'Gericht':gericht})
tokens_grouped = df_raw.groupby(['Datum', 'Gericht'])['tokens']
df_stats = tokens_grouped.describe()
df_stats['5%'] = tokens_grouped.quantile(0.05)
df_stats['95%'] = tokens_grouped.quantile(0.95)
df_stats = df_stats.reset_index()

df_stats_all = df_raw.groupby('Datum')['tokens'].describe()
df_stats_all['5%'] = df_raw.groupby('Datum')['tokens'].quantile(0.05)
df_stats_all['95%'] = df_raw.groupby('Datum')['tokens'].quantile(0.95)
df_stats_all['Gericht'] = 'Alle'
df_stats_all = df_stats_all.reset_index()

df_stats_all = df_stats.append(df_stats_all, sort=True)

In [ ]:
df_std = df_raw.groupby(['Gericht']).std().sort_values('tokens', ascending=False)
df_std.columns = ['Standardabweichung']
df_std = df_std.reset_index()
with open('../tables/makro_decisions_tokens_std.tex', 'w') as f:
    df_std.to_latex(f, index=False)

In [ ]:
diss_data(
    'makro_decisions_tokens_bverfg_std',
    de_num_format(f"{df_raw.groupby(['Gericht']).std().loc['BVerfG']['tokens']:,.2f}")
)

text = format_list([
    f'{gericht}: ' + de_num_format(f'{std:,.2f}')
    for gericht, (std,) in df_raw.groupby(['Gericht']).std().sort_values('tokens', ascending=False).iterrows()
], join_std=', ', join_last=' und ')

diss_data(
    'makro_decisions_tokens_std_text',
    text
)

# Anzahl größerer Entscheidungen pro Jahr und Gericht

In [ ]:
df

In [ ]:
min_size = 3000
df_raw['lang'] = [
    f' Größer als oder gleich {min_size} Tokens'
    if t >= min_size
    else f'Kleiner als {min_size} Tokens'
    for t in df_raw.tokens
]
df = df_raw[(df_raw.Datum <= '2020') & (df_raw.Gericht!="GmSOGB")].groupby(['Gericht', 'Datum', 'lang']).count()

df_all = df.groupby(['Datum', 'lang']).sum()
df_all['Gericht'] = 'Durchschnitt'
df_all.tokens = df_all.tokens/len(set(df.index.get_level_values('Gericht')))

df_all = df_all.reset_index().set_index(['Gericht', 'Datum', 'lang'])

df = df.append(df_all, sort=True)
df = df.reset_index()
df["Row"] = "Entscheidungen"
chart = alt.Chart(df).mark_line(point=True).encode(
    alt.Y('tokens:Q', title=None),
    alt.X('Datum:O'),
    alt.Color(
        'Gericht:N', legend=None
    ),
    alt.Column('lang:N', title=None, spacing=5),
    alt.Row("Row", header=alt.Header(title="Entscheidungen", labels=False, titlePadding=-5)),
    alt.Shape('Gericht:N', legend=None, scale=alt.Scale(range=['square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right','circle'])),
).resolve_scale(y='independent').properties(width=150, height=150).configure_point(size=40)
save_chart(chart, f'makro_decisions_count_spruchkoerper_year_min_max_{min_size}_token')

In [ ]:
legend_chart = alt.Chart(df).mark_point(filled=True, opacity=1, size=0).encode(
    alt.Color('Gericht:N'),
    alt.Shape('Gericht:N', scale=alt.Scale(range=['square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right','circle']))
).configure_view(strokeWidth=0).properties(width=1, height=1, padding={'top': 28}).configure_legend(
    offset=13,  # Adjust distance
)
save_chart(legend_chart, f'makro_decisions_count_spruchkoerper_year_min_max_{min_size}_token_legend')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color(
        'Gericht:N', 
        legend=None,
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(chart_graycolor, f'makro_decisions_count_spruchkoerper_year_min_max_{min_size}_token_graycolor')

In [ ]:
legend_chart_graycolor = legend_chart.encode(
    alt.Color(
        'Gericht:N', 
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(legend_chart_graycolor, f'makro_decisions_count_spruchkoerper_year_min_max_{min_size}_token_legend_graycolor')

In [ ]:
def get_makro_decisions_tokens_boxplot(df_stats_all, colors= [
    'rgb(0, 0, 0)',
    'rgb(76, 120, 168)',
    'rgb(245, 133, 24)',
    'rgb(228, 87, 86)',
    'rgb(114, 183, 178)',
    'rgb(84, 162, 75)',
    'rgb(238, 202, 59)',
    'rgb(178, 121, 162)',
    'rgb(255, 157, 166)',
]):
    return alt.LayerChart(df_stats_all[
        (df_stats_all.Gericht!='GmSOGB') & (df_stats_all.Datum<='2020')
    ]).encode(
        x=alt.X('Datum:O', 
            axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.value : ''"
            )),
        tooltip=['min:Q', '5%:Q', '25%:Q', '50%:Q', '75%:Q', '95%:Q', 'max:Q', 'mean:Q'],
    ).add_layers(
        alt.Chart().mark_rule().encode(y=alt.Y('5%:Q', title='Tokens', axis=alt.Axis(tickCount=8, tickMinStep=20)), y2='95%:Q'),
        alt.Chart().mark_bar(width=10).encode(
            y='25%:Q', 
            y2='75%:Q', 
            color=alt.Color(
                'Gericht:N', 
                legend=None,
                scale=alt.Scale(range=colors)
            )
        ),
        alt.Chart().mark_tick(thickness=2, color='white', width=10).encode(y='50%:Q'),
    #     alt.Chart().mark_tick(color='red', width=15).encode(y='mean:Q'),


    ).properties(
        width=140,
        height=200
    ).facet(
        facet='Gericht:N',
        columns=4,
        spacing=5
    )
chart = get_makro_decisions_tokens_boxplot(df_stats_all)
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_tokens_boxplot')
# Removed outliers, too high to see other growth

In [ ]:
chart = get_makro_decisions_tokens_boxplot(df_stats_all, [
    'black',
])
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_tokens_boxplot_graycolor')
# Removed outliers, too high to see other growth

# Struktur (Seqitems)

In [ ]:
node_docs = [n for n, t in G.nodes(data='type') if t == 'document']
seqitems_count_dict = {n.split('_')[0]: 0 for n in node_docs}
for n, t in G.nodes(data='type'):
    if t == 'seqitem':
        seqitems_count_dict[n.split('_')[0]] += 1

seqitems_count = [seqitems_count_dict[n.split('_')[0]] for n in node_docs]
tokens = [G.nodes[n]['tokens_n'] for n in node_docs]
dates = [G.nodes[n]['datum'][:4] for n in node_docs]
gericht = [G.nodes[n]['gericht'] for n in node_docs]

In [ ]:
df_raw = pd.DataFrame({'Datum': dates, 'Elemente': seqitems_count, 'Gericht':gericht})
elem_grouped = df_raw.groupby(['Datum', 'Gericht'])['Elemente']
df_stats = elem_grouped.describe()
df_stats['5%'] = elem_grouped.quantile(0.05)
df_stats['95%'] = elem_grouped.quantile(0.95)
df_stats = df_stats.reset_index()

df_stats_all = df_raw.groupby('Datum')['Elemente'].describe()
df_stats_all['5%'] = df_raw.groupby('Datum')['Elemente'].quantile(0.05)
df_stats_all['95%'] = df_raw.groupby('Datum')['Elemente'].quantile(0.95)
df_stats_all['Gericht'] = 'Alle'
df_stats_all = df_stats_all.reset_index()

df_stats_all = df_stats.append(df_stats_all, sort=True)

In [ ]:
chart = get_makro_decisions_tokens_boxplot(df_stats_all)
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_seqitems_boxplot')
# Removed outliers, too high to see other growth

In [ ]:
chart = get_makro_decisions_tokens_boxplot(df_stats_all, ["black"])
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_seqitems_boxplot_graycolors')
# Removed outliers, too high to see other growth

In [ ]:
df_raw['nr'] = node_docs
df_raw.sort_values('Elemente', ascending=False).head(20)